# 01 - The ceiling: how well does clone growth even reproduce?

The one notebook that can redirect the project.

Borrowed from Weinreb et al. 2020: establish how well the outcome reproduces
before asking a model to predict it. If clone log growth has reliability R,
then no predictor of single-replicate growth can correlate much above sqrt(R).
A ridge on raw log-CPM already reaches r = 0.441 on held-out clones.

**Two levels of replication, and they are not equally honest.**

- *Within family:* the two EE passages of one BL tumour. Both divide by the
  **same** BL fraction, so the shared denominator inflates their correlation.
  Reported, but flagged.
- *Across families:* independent baselines and independent passages. This is
  the honest reliability, and the number to correct against.

**Decision gate:** if growth sits close to its ceiling at ~0.44, promote
per-gene log fold change to the primary target and demote growth to a
calibration check. Cheap now; expensive after notebook 06.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))
import etracer as et

P = et.Paths(Path.cwd().parent)
manifest = pd.read_csv(P.derived / "clone_manifest.csv")
print(manifest.shape)

In [ ]:
# --- BL -> EE growth, one row per (clone, family, EE passage) -------------
# Start with extinction floored as an observed zero; notebook 02 contrasts
# this against treating it as censored.
g = et.growth_table(manifest, "BL", "EE", extinct="pseudo")
print(f"rows: {len(g)}  clones: {g.clone.nunique()}  EE samples: {g.dst_sample.nunique()}")
print(f"detected at EE: {g.detected_dst.sum()} / {len(g)} ({g.detected_dst.mean():.1%})")
print(f"\nlog2 growth: {g.log2_growth.min():.2f} .. {g.log2_growth.max():.2f}"
      f"  (range {g.log2_growth.max() - g.log2_growth.min():.1f} units)")
g.head()

In [ ]:
# --- MIN CELLS FILTER ----------------------------------------------------
# A clone seen once at BL has a clone fraction with enormous relative error,
# and that error propagates straight into log2 growth. Everything below uses
# this threshold; notebook 02 sweeps it.
MIN_BL_CELLS = 20
gf = g[g.n_cells_src >= MIN_BL_CELLS].copy()
print(f"threshold >= {MIN_BL_CELLS} BL cells: {len(gf)} rows, {gf.clone.nunique()} clones")

In [ ]:
# --- within-family reliability (INFLATED - shared BL denominator) ---------
w = (gf.pivot_table(index=["clone", "family"], columns="branch",
                    values="log2_growth", aggfunc="first")
       .dropna(axis=1, how="all"))
pairs = [c for c in w.columns]
print("EE branches:", pairs)
if len(pairs) >= 2:
    a, b = w[pairs[0]], w[pairs[1]]
    r_within = et.pearson(a, b)
    print(f"\nwithin-family passage-to-passage r = {r_within:.3f}  (n = {(a.notna() & b.notna()).sum()})")
    print("INFLATED: both passages share the same BL fraction in the denominator.")
else:
    r_within = np.nan

In [ ]:
# --- across-family reliability (the honest number) ------------------------
# Average the two passages within a family, then compare families: independent
# baselines, independent passages.
fam_g = (gf.groupby(["clone", "family"])["log2_growth"].mean()
           .unstack("family"))
print("family columns:", list(fam_g.columns))
print(f"clones with >= 2 families: {(fam_g.notna().sum(1) >= 2).sum()}")

rows = []
cols = list(fam_g.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        x, y = fam_g[cols[i]], fam_g[cols[j]]
        n = int((x.notna() & y.notna()).sum())
        rows.append(dict(pair=f"{cols[i]} vs {cols[j]}", n=n,
                         pearson=et.pearson(x, y), spearman=et.spearman(x, y)))
pairwise = pd.DataFrame(rows)
print("\n" + pairwise.to_string(index=False))

In [ ]:
# --- ICC(1): the proper reliability estimate -----------------------------
# One-way random effects over the family-level growth values. This is the
# number to correct against, not the pairwise correlations.
groups = [v[np.isfinite(v)] for v in fam_g.to_numpy()]
icc = et.icc1(groups)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in icc.items()})

R = icc["icc"]
print(f"\nreliability R = {R:.3f}")
print(f"ceiling on a single-replicate predictor, sqrt(R) = {np.sqrt(R):.3f}")

In [ ]:
# --- bootstrap CI over CLONES (not observations) -------------------------
arr = fam_g.to_numpy()
def _icc_boot(idx):
    sub = [v[np.isfinite(v)] for v in arr[idx]]
    return et.icc1(sub)["icc"]

lo, hi = et.bootstrap_ci(_icc_boot, n_items=len(arr), n_boot=1000, seed=0)
print(f"ICC = {R:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print(f"ceiling sqrt(R) 95% CI [{np.sqrt(max(lo,0)):.3f}, {np.sqrt(max(hi,0)):.3f}]")

In [ ]:
# --- what the headroom actually is ---------------------------------------
# r_obs is what a model scores against single-replicate growth.
# r_corrected = r_obs / sqrt(R) is what it would score against noise-free growth.
# If r_corrected approaches 1.0, the predictor has captured all available signal.
MEASURED = {
    "rung 0: clone-identity lookup (family hold-out)": 0.605,
    "rung 1: 5 state fractions": 0.323,
    "pseudobulk mean, 400 genes": 0.376,
    "kernel mean embedding of the cell distribution": 0.441,
}
tab = pd.DataFrame([
    dict(predictor=k, r_obs=v, r_corrected=et.attenuation_correct(v, R),
         pct_of_ceiling=v / np.sqrt(R))
    for k, v in MEASURED.items()
])
print(f"reliability R = {R:.3f}, ceiling = {np.sqrt(R):.3f}\n")
print(tab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
# --- score rung 1 here, from scratch, against this ceiling ---------------
# The only rung that needs no expression matrix: the five state fractions are
# in the metadata. Repeated grouped CV, grouped by clone.
feat = et.state_features(manifest, "BL")
d = (gf.groupby(["clone", "family"])["log2_growth"].mean().rename("y").reset_index()
       .merge(feat, on=["clone", "family"], how="inner"))
Xcols = [f"state_{s}" for s in et.STATES]
X, y, grp = d[Xcols].to_numpy(), d["y"].to_numpy(), d["clone"].to_numpy()
print(f"design: {X.shape}, clones: {len(np.unique(grp))}")

scores = et.grouped_cv_r(X, y, grp, n_repeats=20, seed=0)
null = et.permutation_null_r(X, y, grp, n_perm=20)
print(f"\nrung 1, grouped 5-fold CV x 20: r = {np.nanmean(scores):.3f} +/- {np.nanstd(scores):.3f}")
print(f"permutation null:                r = {np.nanmean(null):.3f} +/- {np.nanstd(null):.3f}")
print(f"as a share of the ceiling: {np.nanmean(scores) / np.sqrt(R):.1%}")
print(f"\nper-repeat spread: {np.nanmin(scores):.3f} .. {np.nanmax(scores):.3f}"
      "   <- why single splits are not reportable")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].scatter(fam_g.iloc[:, 0], fam_g.iloc[:, 1], s=30, alpha=0.7)
lim = np.nanpercentile(fam_g.to_numpy(), [1, 99])
ax[0].plot(lim, lim, "k--", lw=1, alpha=0.5)
ax[0].set_xlabel(f"log2 growth, {fam_g.columns[0]}")
ax[0].set_ylabel(f"log2 growth, {fam_g.columns[1]}")
ax[0].set_title(f"same clone, two founder families\nr = {pairwise.pearson.iloc[0]:.3f}")
ax[0].grid(alpha=0.25)

names = list(MEASURED)
vals = [MEASURED[k] for k in names]
ax[1].barh(range(len(names)), vals, alpha=0.85)
ax[1].axvline(np.sqrt(R), color="crimson", ls="--",
              label=f"ceiling sqrt(R) = {np.sqrt(R):.2f}")
ax[1].axvline(np.nanmean(null), color="grey", ls=":", label="permutation null")
ax[1].set_yticks(range(len(names)))
ax[1].set_yticklabels([n.split(":")[-1].strip()[:34] for n in names], fontsize=8)
ax[1].set_xlabel("out-of-fold Pearson r")
ax[1].set_title("measured rungs against the ceiling")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.25, axis="x")
plt.tight_layout(); plt.show()

In [ ]:
# --- save -----------------------------------------------------------------
out = dict(min_bl_cells=MIN_BL_CELLS, icc=R, icc_lo=lo, icc_hi=hi,
           ceiling=float(np.sqrt(R)), r_within_family=r_within,
           rung1_r=float(np.nanmean(scores)), rung1_sd=float(np.nanstd(scores)),
           null_r=float(np.nanmean(null)))
pd.Series(out).to_csv(P.derived / "reliability.csv", header=False)
pairwise.to_csv(P.derived / "reliability_pairwise.csv", index=False)
tab.to_csv(P.derived / "headroom.csv", index=False)
print(pd.Series(out).to_string())

## Decision gate

- reliability R (ICC, with CI):
- ceiling sqrt(R):
- best measured rung as a share of the ceiling:

**If the ceiling is near 0.44**, growth is close to saturated. Promote per-gene
log fold change to primary, keep growth as a calibration check, and rerun this
notebook's reliability calculation on per-gene LFC once the expression matrix
is converted - that target may have far more headroom.

**If the ceiling is well above 0.44**, growth stays primary and there is real
room for the conditioning ladder to climb.

One caveat to carry either way: the ICC is computed over clones that survive
the `MIN_BL_CELLS` filter and appear in at least two families, so it describes
the clones you can actually model, not the whole repertoire.